In [2]:
from keras.models import Sequential
from keras.layers import Conv2D, ZeroPadding2D, Activation, Input, concatenate, Layer
from keras.models import Model
from keras.layers import BatchNormalization, MaxPooling2D, AveragePooling2D, Concatenate, Lambda, Flatten, Dense
from keras.initializers import glorot_uniform
from keras import backend as K

K.set_image_data_format('channels_first')
# import cv2
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from fr_utils import *
from inception_blocks_v2 import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Encoding face images into a 128-dimensional vector

In [3]:
FRmodel = faceRecoModel(input_shape=(3, 96, 96))

E0000 00:00:1787990584.249397    5284 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1787990584.250194    5411 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1787990584.290784    5284 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
print("Total Params:", FRmodel.count_params())

Total Params: 3743280


Triplet Loss

In [5]:
def triplet_loss(y_true, y_pred, alpha=0.2):
    """
    Implementation of the triplet loss as defined by the triplet loss formula

    Arguments:
    y_true -- true labels, required when you define a loss in keras, you don't need it in this function
    y_pred -- python list containing three objects:
            anchor -- the encodings for the anchor images, of shaoe (None, 128)
            positive -- the encodings for the positive images, of shape (None, 128)
            negative -- the encodings for the negative images, of shape (None, 128)
    Returns:
    loss -- real number, value of the loss
    """

    anchor, positive, negative = y_pred[0], y_pred[1], y_pred[2]

    pos_dist = tf.reduce_sum(tf.square(anchor, positive), axis=-1)

    neg_dist = tf.reduce_sum(tf.square(anchor, negative), axis=-1)

    basic_loss = pos_dist - neg_dist + alpha

    loss = tf.reduce_sum(tf.maximum(basic_loss, 0.0))

    return loss

In [6]:
y_true = (None, None, None)
y_pred = (tf.random.normal([3, 128], mean=6, stddev=0.1, seed=1),
          tf.random.normal([3, 128], mean=1, stddev=1, seed=1),
          tf.random.normal([3, 128], mean=3, stddev=4, seed=1))

loss = triplet_loss(y_true, y_pred)

print('loss = ' + str(loss.numpy()))

loss = 0.6


Loading the pre-trained model

In [7]:
FRmodel.compile(optimizer='adam', loss=triplet_loss, metrics=['accuracy'])
load_weights_from_FaceNet(FRmodel)

KeyError: 'inception_3c_3x3_conv2_w'